<img src='https://gitlab.eumetsat.int/eumetlab/oceans/ocean-training/tools/frameworks/-/raw/main/img/MSOC_banner.png' align='right' width='100%'/>

<a href="../Index.ipynb">Index</a> | <a href="./Access_OLCI_EUMETSAT_Data_Store.ipynb">Accessing Sentinel-3 OLCI data through the EUMETSAT Data Store</a> | <a href="./Access_PACE_EarthData.ipynb">Accessing PACE OCI through EarthData</a> | <a href="./Access_multisensor_CMEMS_Data_Store.ipynb">Accessing multisensor data through the CMEMS Data Store</a> 

**Copyright:** 2026 European Union <br>
**License:** MIT <br>
**Authors:** Ben Loveday (Innoflair for EUMETSAT), Javier A. Concha (Serco for EUMETSAT)

<div class="alert alert-block alert-success">
<h3>Multi-sensor Ocean Colour Course</h3></div>

<div class="alert alert-block alert-warning">
    
<b>PREREQUISITES </b>
    
This notebook has the following prerequisites:
- **<a href="https://identity.dataspace.copernicus.eu/auth/realms/CDSE/login-actions/registration?client_id=account-console&tab_id=7mxjCv4mJxw" target="_blank">A Copernicus Data Space Ecosystem account</a>** if you are using or plan to use the CDSE.

There are no prerequisite notebooks for this module.
</div>
<hr>

# Accessing Copernicus Sentinel-2 MSI data via the Copernicus Data Space Ecosystem (CDSE)

### Data used

| Dataset | CDSE collection ID | CDSE collection<br>description | WEkEO dataset ID | WEkEO description |
|:--------------------:|:-----------------------:|:-------------:|:-----------------:|:-----------------:|
| Sentinel-2 Level 1C Top of Atmosphere | sentinel-2-l1c | <a href="https://documentation.dataspace.copernicus.eu/Data/SentinelMissions/Sentinel2.html#sentinel-2-level-1c-top-of-atmosphere-toa" target="_blank">Description</a> | EO:ESA:DAT:SENTINEL-2 | <a href="https://wekeo.copernicus.eu/data?view=dataset&dataset=EO:ESA:DAT:SENTINEL-2" target="_blank">Description</a> |
| Sentinel-2 Level 2A Surface Reflectance | sentinel-2-level-2a-surface-reflectance | <a href="https://documentation.dataspace.copernicus.eu/Data/SentinelMissions/Sentinel2.html#sentinel-2-level-2a-surface-reflectance" target="_blank">Description</a> | EO:ESA:DAT:SENTINEL-2 | <a href="https://wekeo.copernicus.eu/data?view=dataset&dataset=EO:ESA:DAT:SENTINEL-2" target="_blank">Description</a> |

### Learning outcomes

At the end of this notebook you will know;
* how to search and download data from the Copernicus Data Space Ecosystem using the <font color="#138D75">**OData**</font> API
* How to refine your <font color="#138D75">**searches**</font> for Copernicus Sentinel-2 MSI marine products
* How to refine your searches for temporal, spatial, orbital and tile coverage parameters
* How to <font color="#138D75">**download**</font> full products

### Outline

The Copernicus Data Space Ecosystem (CDSE) provides access to all Copernicus Sentinel-2 MSI data, including the most up to date products. The CDSE offers a variety of interfaces, including a web GUI and suite of APIs to support automated, programmatic access. In this notebook, we will show you how to use the <a href="https://documentation.dataspace.copernicus.eu/APIs/OData.html" target="_blank">OData</a> APIs to search for, filter and download MSI data.

<hr>

<div class="alert alert-info" role="alert">

## Importing dependencies

</div>

We begin by importing all of the libraries that we need to run this notebook. If you have built your python using the environment file provided in this repository, then you should have everything you need. For more information on building environment, please see the repository **<a href="../README.md" target="_blank">README</a>**.

In [1]:
import datetime               # a library that supports the creation of date objects
import os                     # a library that allows us access to basic operating system commands like making directories
import shutil                 # a library that allows us access to basic operating system commands like copy
import zipfile                # a library that allows us to unzip zip-files.
import eumdac                 # a tool that helps us download via the eumetsat/data-store
from pathlib import Path      # a library that helps construct system path objects
import getpass                # a library to help us enter passwords
import requests               # a library that helps us to make HTTP requests

Next we will create a download directory to store the products we will download in this notebook.

In [2]:
download_dir = os.path.join(os.path.dirname(os.getcwd()), "Data", "Preprepared", "Day1", "Data_access", "MSI")
os.makedirs(download_dir, exist_ok=True)

<div class="alert alert-info" role="alert">

##  Downloading from the Copernicus Data Space Ecosystem (CDSE) using the OData API

</div>

<div class="alert alert-block alert-success">

### Accessing the CDSE

To download Copernicus Sentinel-2 MSI products from the CDSE, we will use the <a href="https://documentation.dataspace.copernicus.eu/APIs/OData.html" target="_blank">OData API</a>. OData does not require any special packages for downloading, we will rely on the Python `requests` package. To download data from the CDSE OData API, you need to provide credentials. To obtain these you should first register at for a <a href="https://identity.dataspace.copernicus.eu/auth/realms/CDSE/login-actions/registration?client_id=account-console&tab_id=7mxjCv4mJxw" target="_blank">Copernicus Data Space Ecosystem account</a> and note your `username` and `password` for use below.
</div>

In order to use the OData API, we need to provide some information on the expected API endpoints for authorisation, cataloging and downloading, as below;

In [3]:
auth_url = "https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token"
catalog_url = "https://catalogue.dataspace.copernicus.eu/odata/v1/Products"
download_url = "https://zipper.dataspace.copernicus.eu/odata/v1/Products"

Lets create our credentials file, if it doesn't already exist, and read our credentials from it.

In [4]:
# load credentials
cdse_credentials_file = Path(Path.home() / '.cdse' / '.cdse_credentials')

if os.path.exists(cdse_credentials_file):
    username, password = Path(cdse_credentials_file).read_text().split(',')
else:
    # creating authentication file
    username = input('Enter your username: ')
    password = getpass.getpass('Enter your password: ')
    try:
        os.makedirs(os.path.dirname(cdse_credentials_file), exist_ok=True)
        with open(cdse_credentials_file, "w") as f:
            f.write(f'{username},{password}')
    except:
        pass

Now we will use these credentials to create a token to interface with the CDSE...

In [5]:
data = {
    "client_id": "cdse-public",
    "grant_type": "password",
    "username": username,
    "password": password
}

token = requests.post(auth_url, data=data).json()["access_token"]
print(f"Access token: {token[0:30]}...")

Access token: eyJhbGciOiJSUzI1NiIsInR5cCIgOi...


Now we have a token, we can make requests from the CDSE. Lets start forming our request by setting the required collection and product type. In this case, the collection will be `Sentinel-2` in all cases, but the product type should switch between `S2MSI1C` and `S2MSI2A` as required.

In [6]:
collection = 'SENTINEL-2'
product_type = 'S2MSI1C'

Typically, we want to search by time, space, orbital parameters and tile coverage. We'll configure all of these as options below. Lets start by setting some time bounds. We can do this using Python datetime objects, as follow;

In [7]:
# time filter the collection for matching products
dtstart = datetime.datetime(2025, 8, 13, 0, 0)
dtend = datetime.datetime(2025, 8, 14, 0, 0)

Now lets determine a spatial region. We can do this by defining a polygon of any shape using a WKT format, e.g. 'POLYGON ((10.09 56.09, 10.34 56.09, 10.34 56.19, 10.09 56.19, 10.09 56.09))'. Lets define one using a regular box....

In [8]:
west = 2.0
south = 54.0
east = 3.0
north = 55.0

roi = [[west, south], [west, north], [east, north], [east, south], [west, south]]
geo = 'POLYGON(({}))'.format(','.join(["{} {}".format(*coord) for coord in roi]))

Our time and space parameters are now set, lets create our search query and see how many products correspond...

In [9]:
headers = {"Authorization": f"Bearer {token}"}
params = {
    "$filter": (
        f"Collection/Name eq '{collection}' "
        f"and ContentDate/Start ge {dtstart.strftime('%Y-%m-%dT%H:%M:%S.000Z')} "
        f"and ContentDate/Start le {dtend.strftime('%Y-%m-%dT%H:%M:%S.000Z')} "
        f"and Attributes/OData.CSC.StringAttribute/any(a:a/Name eq 'productType' and a/Value eq '{product_type}') "
        f"and OData.CSC.Intersects(area=geography'SRID=4326;{geo}')"
    ),
    "$top": 100
}

resp = requests.get(catalog_url, headers=headers, params=params)
products = resp.json()["value"]

print(f"Found {len(products)} matching products")

Found 5 matching products


MSI granules are quite small, and are mapped to UTM tiles. To better filter the outputs we want to download, we can refine our query by using the relative orbit and UTM tile of the product list, as below;

In [10]:
rel_orbit_match = "R137"
tile_match = "T31UDV"

Lets use simple filename matching to see how many of our products match ther required orbit and UTM parameters.

In [11]:
filtered_products = []
for product, num in zip(products,range(1,len(products)+1)):
    if tile_match in product['Name'] and rel_orbit_match in product["Name"]:
        print(f"{str(num).zfill(2)}: {product['Name']} <<< MATCH >>>")
        filtered_products.append(product)
    else:
        print(f"{str(num).zfill(2)}: {product['Name']}")

01: S2B_MSIL1C_20250813T110619_N0511_R137_T31UEB_20250813T130513.SAFE
02: S2B_MSIL1C_20250813T110619_N0511_R137_T31UDB_20250813T130513.SAFE
03: S2B_MSIL1C_20250813T110619_N0511_R137_T31UDA_20250813T130513.SAFE
04: S2B_MSIL1C_20250813T110619_N0511_R137_T31UEA_20250813T130513.SAFE
05: S2B_MSIL1C_20250813T110619_N0511_R137_T31UDV_20250813T130513.SAFE <<< MATCH >>>


Finally, lets download and unzip the products that match our filtered list...

In [12]:
for product in filtered_products:
    print(f"Downloading {product['Name']}")
    with requests.get(f"{download_url}({product['Id']})/$value", headers=headers, stream=True) as r:
        r.raise_for_status()
        with open(os.path.join(download_dir, f"{product['Name']}.zip"), "wb") as fdst:
            for chunk in r.iter_content(chunk_size=8192):
                fdst.write(chunk)
        print(f"✅ Download complete: {product['Name']}")

    with zipfile.ZipFile(fdst.name, 'r') as zip_ref:
        for file in zip_ref.namelist():
            if file.startswith(product["Name"]):
                zip_ref.extract(file, download_dir)
        print(f'✅ Unzip complete: {fdst.name}')
        
    os.remove(fdst.name)

✅ Download complete: S2B_MSIL1C_20250813T110619_N0511_R137_T31UDV_20250813T130513.SAFE
✅ Unzip complete: /Users/benloveday/Code/git_repositories/CMTS/internal/ocean/applications/ocean-colour-applications/3_courses/msoc/Data/Preprepared/Day1/Data_access/MSI/S2B_MSIL1C_20250813T110619_N0511_R137_T31UDV_20250813T130513.SAFE.zip


<div class="alert alert-success" role="alert">

## What next?

You should be able to adapt the methods above to search the CDSE for the Copernicus Sentinel-2 MSI level-1B and level-2 data you require for this course, using your project ROI and time bounds information as input. If you need more information on using `OData` to interface with the CDSE you can find a comprehensive overview in the <a href="https://documentation.dataspace.copernicus.eu/APIs/OData.html" target="_blank">online manual</a>.

</div>

<hr>
<a href="../Index.ipynb">Index</a> | <a href="./Access_OLCI_EUMETSAT_Data_Store.ipynb">Accessing Sentinel-3 OLCI data through the EUMETSAT Data Store</a> | <a href="./Access_PACE_EarthData.ipynb">Accessing PACE OCI through EarthData</a> | <a href="./Access_multisensor_CMEMS_Data_Store.ipynb">Accessing multisensor data through the CMEMS Data Store</a> 
<hr>
<a href="https://gitlab.eumetsat.int/eumetlab/ocean">View on GitLab</a> | <a href="https://training.eumetsat.int/">EUMETSAT Training</a> | <a href=mailto:ops@eumetsat.int>Contact helpdesk for support </a> | <a href=mailto:.training@eumetsat.int>Contact our training team to collaborate on and reuse this material</a></span></p>